# Model — original → edited (brightness) translation

A **residual U-Net**: input a grayscale original image, predict a per-pixel brightness *residual*, and the output is `input + residual` (clamped to [0,1]). Learning the residual is easy here because most pixels barely change.

- Grayscale, single channel. `DOWNSCALE` controls the working resolution (default 144×96).
- The U-Net reflect-pads any input up to a multiple of 16 internally and crops the output back, so **any `DOWNSCALE` size works** even if it is not divisible by 16.
- Loss: L1. Metrics on test: relative-MAE (your metric), PSNR, SSIM.

Requires: `pip install torch torchvision` (and `scikit-image` for SSIM).

In [ ]:
import os
import random
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---- Config ----
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
SPLIT_DIR = os.path.join(DATA_DIR, "720*480")   # contains train/ and test/
TRAIN_DIR = os.path.join(SPLIT_DIR, "train")
TEST_DIR = os.path.join(SPLIT_DIR, "test")
CKPT_PATH = os.path.join(os.getcwd(), "best_model.pt")

DOWNSCALE = (144, 96)    # (width, height) fed to the model; None = full 720x480
BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-3
BASE_CH = 32             # U-Net width; lower = faster/lighter
VAL_RATIO = 0.1          # fraction of train held out for early stopping
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE)

## Dataset

Loads paired grayscale images, scales pixels to [0,1], returns `(original, edited, filename)`. Optional paired horizontal-flip augmentation (same flip applied to both, so alignment is preserved).

In [ ]:
class PairedDataset(Dataset):
    def __init__(self, root, files, downscale=None, augment=False):
        self.root = root
        self.files = files
        self.downscale = downscale   # (w, h) or None
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def _load(self, folder, f):
        im = Image.open(os.path.join(self.root, folder, f)).convert("L")
        if self.downscale is not None:
            im = im.resize(self.downscale, Image.LANCZOS)
        return im

    def __getitem__(self, i):
        f = self.files[i]
        o = self._load("original", f)
        e = self._load("edited", f)
        if self.augment and random.random() < 0.5:
            o = o.transpose(Image.FLIP_LEFT_RIGHT)
            e = e.transpose(Image.FLIP_LEFT_RIGHT)
        o = torch.from_numpy(np.asarray(o, np.float32) / 255.0).unsqueeze(0)  # (1,H,W)
        e = torch.from_numpy(np.asarray(e, np.float32) / 255.0).unsqueeze(0)
        return o, e, f

In [ ]:
def list_pairs(root):
    o = set(f for f in os.listdir(os.path.join(root, "original")) if f.lower().endswith(".jpg"))
    e = set(f for f in os.listdir(os.path.join(root, "edited")) if f.lower().endswith(".jpg"))
    return sorted(o & e)

train_all = list_pairs(TRAIN_DIR)
test_files = list_pairs(TEST_DIR)

# Hold out a validation slice from train.
rng = random.Random(SEED)
shuffled = train_all[:]
rng.shuffle(shuffled)
n_val = int(len(shuffled) * VAL_RATIO)
val_files = sorted(shuffled[:n_val])
train_files = sorted(shuffled[n_val:])
print(f"train: {len(train_files)} | val: {len(val_files)} | test: {len(test_files)}")

train_ds = PairedDataset(TRAIN_DIR, train_files, DOWNSCALE, augment=True)
val_ds = PairedDataset(TRAIN_DIR, val_files, DOWNSCALE, augment=False)
test_ds = PairedDataset(TEST_DIR, test_files, DOWNSCALE, augment=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

## Residual U-Net

Depth-4 U-Net (downsamples by 16). To allow **any** input size, `forward` reflect-pads the input up to a multiple of 16, runs the network, then crops the residual back to the original size. GroupNorm keeps small batches stable. Final image = `input + residual`, clamped to [0,1].

In [ ]:
def gn(c):
    return nn.GroupNorm(num_groups=min(8, c), num_channels=c)


class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), gn(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), gn(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class ResUNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.d1 = DoubleConv(1, b)
        self.d2 = DoubleConv(b, b * 2)
        self.d3 = DoubleConv(b * 2, b * 4)
        self.d4 = DoubleConv(b * 4, b * 8)
        self.pool = nn.MaxPool2d(2)
        self.bott = DoubleConv(b * 8, b * 16)
        self.up4 = nn.ConvTranspose2d(b * 16, b * 8, 2, stride=2)
        self.u4 = DoubleConv(b * 16, b * 8)
        self.up3 = nn.ConvTranspose2d(b * 8, b * 4, 2, stride=2)
        self.u3 = DoubleConv(b * 8, b * 4)
        self.up2 = nn.ConvTranspose2d(b * 4, b * 2, 2, stride=2)
        self.u2 = DoubleConv(b * 4, b * 2)
        self.up1 = nn.ConvTranspose2d(b * 2, b, 2, stride=2)
        self.u1 = DoubleConv(b * 2, b)
        self.out = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        inp = x
        h, w = x.shape[-2:]
        m = 16
        ph, pw = (m - h % m) % m, (m - w % m) % m
        if ph or pw:
            x = F.pad(x, (0, pw, 0, ph), mode="reflect")  # pad right/bottom to multiple of 16
        c1 = self.d1(x)
        c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2))
        c4 = self.d4(self.pool(c3))
        bn = self.bott(self.pool(c4))
        y = self.u4(torch.cat([self.up4(bn), c4], 1))
        y = self.u3(torch.cat([self.up3(y), c3], 1))
        y = self.u2(torch.cat([self.up2(y), c2], 1))
        y = self.u1(torch.cat([self.up1(y), c1], 1))
        residual = self.out(y)[..., :h, :w]              # crop back to original size
        return torch.clamp(inp + residual, 0.0, 1.0)     # residual learning


model = ResUNet(BASE_CH).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResUNet params: {n_params/1e6:.2f}M")

## Training (L1 loss, early stopping on validation)

In [ ]:
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for o, e, _ in loader:
            o, e = o.to(DEVICE), e.to(DEVICE)
            pred = model(o)
            loss = criterion(pred, e)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * o.size(0); n += o.size(0)
    return total / max(n, 1)


best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_dl, True)
    va = run_epoch(val_dl, False)
    flag = ""
    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), CKPT_PATH)
        flag = "  <- saved best"
    print(f"epoch {epoch:03d} | train L1 {tr:.4f} | val L1 {va:.4f}{flag}")
print("best val L1:", best_val)

## Evaluate on test set

Reports the identity baseline (original vs edited, no model) and the model's predictions vs edited, using relative-MAE, PSNR, and SSIM. The model should clearly beat the identity baseline.

In [ ]:
try:
    from skimage.metrics import structural_similarity
    HAS_SSIM = True
except Exception:
    HAS_SSIM = False
    print("scikit-image not installed; SSIM will be skipped.")


def metrics_np(pred, target):
    """pred, target: float arrays in [0,255]."""
    rmae = float(np.sum(np.abs(pred - target)) / (np.sum(target) + 1e-8))
    mse = float(np.mean((pred - target) ** 2))
    psnr = float("inf") if mse == 0 else float(20 * np.log10(255.0) - 10 * np.log10(mse))
    ssim = structural_similarity(target, pred, data_range=255.0) if HAS_SSIM else float("nan")
    return rmae, psnr, ssim


model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

acc_model = np.zeros(3); acc_base = np.zeros(3); count = 0
with torch.no_grad():
    for o, e, _ in test_dl:
        o = o.to(DEVICE)
        pred = model(o).cpu().numpy()[0, 0] * 255.0
        orig = o.cpu().numpy()[0, 0] * 255.0
        targ = e.numpy()[0, 0] * 255.0
        acc_model += np.array(metrics_np(pred, targ))
        acc_base += np.array(metrics_np(orig, targ))
        count += 1

am, ab = acc_model / count, acc_base / count
print(f"tested {count} images")
print(f"{'':10s} {'rel_MAE':>10s} {'PSNR':>10s} {'SSIM':>10s}")
print(f"{'baseline':10s} {ab[0]:10.4f} {ab[1]:10.3f} {ab[2]:10.4f}")
print(f"{'model':10s} {am[0]:10.4f} {am[1]:10.3f} {am[2]:10.4f}")

## Qualitative check — original | predicted | edited

In [ ]:
import matplotlib.pyplot as plt

K = 5
sample = test_files[:K]
vis_ds = PairedDataset(TEST_DIR, sample, DOWNSCALE, augment=False)
model.eval()
fig, axes = plt.subplots(len(sample), 3, figsize=(12, 3 * len(sample)))
if len(sample) == 1:
    axes = axes[None, :]
with torch.no_grad():
    for row in range(len(sample)):
        o, e, f = vis_ds[row]
        pred = model(o.unsqueeze(0).to(DEVICE)).cpu()[0, 0].numpy()
        for col, (img, title) in enumerate([
            (o[0].numpy(), f"original — {f}"), (pred, "predicted"), (e[0].numpy(), "edited")]):
            axes[row, col].imshow(img, cmap="gray", vmin=0, vmax=1)
            axes[row, col].set_title(title); axes[row, col].axis("off")
plt.tight_layout(); plt.show()